# Kombinle modeli eğitimi (2. sürüm, büyük model)

Bu defter, Kombinle'nin telefonda internetsiz çalışan kıyafet tanıma modelini eğitir.

**Ne öğreniyor?**
- **Türü (~40 tür):** tişört, bluz, tunik, kazak, hırka, blazer, trençkot, yağmurluk, kot, tayt, etek, elbise, tulum…
  ayakkabıda **bot, sandalet, topuklu**; takıda **küpe, kolye, bileklik**; çanta, gözlük, şapka, saat, kemer, çorap, sütyen…
- **"Kıyafet değil":** yemek, araba, kedi, telefon gibi kıyafet olmayan fotoğrafları tanır (uygulama 🫪 gösterir, buluta hiç gitmez).
- **Deseni:** düz, çizgili, kareli, çiçekli, puantiyeli, hayvan deseni ya da başka desen.
- **Kol boyu:** kolsuz, kısa, truvakar, uzun kollu.
- **Etek/elbise/kaban boyu:** mini, diz üstü, diz, midi, maksi.

Renk modelde yok; uygulama rengi fotoğraftan kendisi hesaplıyor (daha tutarlı).
Boydan fotoğraftaki parçaları ayırmak ayrı defterin işi: `kombinle_parca_egitimi.ipynb`.

**Nasıl çalıştırılır?**
1. Üst menü → **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. **Çalışma zamanı → Tümünü çalıştır**.
3. İlk hücre Google Drive izni ister: **izin ver**. Eğitim her dönem sonunda Drive'a kaydedilir;
   bağlantı koparsa yeniden **Tümünü çalıştır** deyince kaldığı yerden devam eder.
4. Yaklaşık **3–4 saat** sürer. Sekmeyi açık bırak.
5. Son hücre bir GitHub anahtarı ister ve modeli Kombinle reposuna gönderir. GitHub yeni APK'yı kendisi derler.

**Veri (ikisi de ticari kullanıma uygun):**
- [Fashionpedia](https://fashionpedia.github.io/home/): etiketler CC BY 4.0. Ticari kullanıma kapalı (NonCommercial) fotoğraflar eğitime alınmıyor.
- [Open Images](https://storage.googleapis.com/openimages/web/index.html) (Google): etiketler CC BY 4.0, fotoğraflar CC BY 2.0.

Uygulamada atıf "Lisanslar" bölümünde yer alıyor.

In [ ]:
# ---------- Ayarlar ----------
import os, sys, json, re, random, time, glob, math, shutil, subprocess, zipfile, csv
from collections import Counter, defaultdict

REPO = 'aliyuksekal/kombinle'        # modelin gönderileceği repo
DAL = 'main'
TABAN = 'EfficientNetV2B2'           # güçlü ve telefonda hâlâ hızlı (~10 MB). Daha küçük istersen: 'MobileNetV3Large'
GIRDI = 260                          # modelin gördüğü resim boyutu (piksel)
KAYIT_BOYU = 300                     # kırpılmış örneklerin diske yazıldığı boyut
SINIF_BASI_EN_FAZLA = 4000           # Fashionpedia: her türden en çok kaç örnek (dengeli eğitim)
SINIF_BASI_EN_AZ = 150               # bundan az örneği olan tür modele alınmaz
OI_SINIF_BASI = 2500                 # Open Images: her türden en çok kaç örnek
OI_DOGRULAMA_BASI = 300
OI_YOK_TOPLAM = 6000                 # "kıyafet değil" örnekleri
EN_KUCUK_KUTU = 48                   # bundan küçük (piksel) kıyafet kutuları atlanır
DONEM_ISINMA = 3                     # 1. aşama: yalnız yeni katmanlar eğitilir
DONEM_INCE = 12                      # 2. aşama: modelin üst katmanları da ince ayarlanır
TOPLU = 48
TOHUM = 7

SINAMA = os.environ.get('KB_SINAMA') == '1'   # yalnız geliştirici sınaması: sahte küçük veri, CPU
if SINAMA:
    GIRDI, KAYIT_BOYU, TOPLU = 96, 112, 16
    TABAN = os.environ.get('KB_TABAN', TABAN)
CALISMA = '/content/kb' if os.path.isdir('/content') and not SINAMA else os.path.abspath(os.environ.get('KB_CALISMA', 'kb'))
VERI = os.path.join(CALISMA, 'fashionpedia')
OI = os.path.join(CALISMA, 'openimages')
KIRPIK = os.path.join(CALISMA, 'kirpik2_%d' % KAYIT_BOYU)   # eski defterin kırpıklarıyla karışmasın
CIKTI = os.path.join(CALISMA, 'cikti')
for d in (VERI, OI, KIRPIK, CIKTI):
    os.makedirs(d, exist_ok=True)
random.seed(TOHUM)
print('Çalışma klasörü:', CALISMA, '(sınama)' if SINAMA else '')

# Google Drive: eğitim her dönem sonunda buraya kaydedilir. Colab bağlantısı koparsa
# "Tümünü çalıştır" ile yeniden başlatınca kaldığı yerden devam eder.
KALICI = None
if os.path.isdir('/content') and not SINAMA:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        KALICI = '/content/drive/MyDrive/kombinle_egitim'
    except Exception as e:
        print('Drive bağlanmadı (sorun değil, ara kayıtlar yalnız bu oturumda tutulur):', e)
KALICI = KALICI or os.path.join(CALISMA, 'kalici')
os.makedirs(KALICI, exist_ok=True)
print('Ara kayıtlar:', KALICI)

In [ ]:
# ---------- Ekran kartı kontrolü ----------
try:
    import tensorflow as tf
    import keras
    gpu = tf.config.list_physical_devices('GPU')
    print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', gpu[0].name if gpu else 'YOK')
    if gpu:
        keras.mixed_precision.set_global_policy('mixed_float16')   # T4'te ~2 kat hızlı; telefona float32 gider
    elif not SINAMA:
        print('\n⚠️  GPU seçilmemiş. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.\n'
              '   GPU olmadan eğitim çok uzun sürer.')
    if not hasattr(tf, 'lite'):
        print('⚠️  Bu TensorFlow sürümünde tf.lite yok; son adımda telefon modeli üretilemez.')
except ImportError:
    tf = None
    print('TensorFlow yok (yalnız veri hazırlama sınanabilir).')

## 1. Fashionpedia'yı indir
Eğitim fotoğrafları yaklaşık 3,5 GB. İndirme ve açma 10–15 dakika sürer.

In [ ]:
ADRESLER = {
    'train2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/train2020.zip',
    'val_test2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip',
    'instances_attributes_train2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_train2020.json',
    'instances_attributes_val2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json',
}


def indir(ad, adres, klasor=None):
    hedef = os.path.join(klasor or VERI, ad)
    if os.path.exists(hedef) and os.path.getsize(hedef) > 0:
        print('var:', ad)
        return hedef
    print('indiriliyor:', ad)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', hedef + '.part', adres], check=True)
    os.replace(hedef + '.part', hedef)
    return hedef


def ac(zip_yolu, klasor):
    if os.path.isdir(klasor) and os.listdir(klasor):
        print('açık:', os.path.basename(klasor))
        return
    print('açılıyor:', os.path.basename(zip_yolu))
    with zipfile.ZipFile(zip_yolu) as z:
        z.extractall(VERI)


FP_KATEGORI = ['shirt, blouse', 'top, t-shirt, sweatshirt', 'sweater', 'cardigan', 'jacket', 'vest', 'pants', 'shorts', 'skirt',
               'coat', 'dress', 'jumpsuit', 'cape', 'glasses', 'hat', 'headband, head covering, hair accessory', 'tie', 'glove',
               'watch', 'belt', 'leg warmer', 'tights, stockings', 'sock', 'shoe', 'bag, wallet', 'scarf', 'umbrella', 'hood',
               'collar', 'lapel', 'epaulette', 'sleeve', 'pocket', 'neckline', 'buckle', 'zipper']


def sahte_veri_kur():
    """Geliştirici sınaması: Fashionpedia biçiminde küçük sahte veri (çokgen maskeler, kol parçaları, boy özellikleri)."""
    from PIL import Image, ImageDraw
    oz = [('plain (pattern)', 'textile pattern'), ('stripe', 'textile pattern'), ('check', 'textile pattern'), ('floral', 'textile pattern'),
          ('dot', 'textile pattern'), ('leopard', 'textile pattern'), ('abstract', 'textile pattern'), ('jeans', 'nickname'),
          ('maxi (length)', 'length'), ('mini (length)', 'length'), ('knee (length)', 'length'), ('midi', 'length'),
          ('above-the-knee (length)', 'length'), ('sleeveless', 'length'), ('short (length)', 'length'), ('elbow-length', 'length'),
          ('wrist-length', 'length'), ('hoodie', 'nickname'), ('trench (coat)', 'nickname'), ('blazer', 'nickname'),
          ('leggings', 'nickname'), ('tunic (top)', 'nickname'), ('denim', 'textile material')]
    giysi = [i for i, k in enumerate(FP_KATEGORI) if FP_KATEGORI.index('umbrella') > i]
    kol_kat = FP_KATEGORI.index('sleeve')
    for bolum, klasor, n in (('train2020', 'train', 320), ('val2020', 'test', 80)):
        os.makedirs(os.path.join(VERI, klasor), exist_ok=True)
        resimler, notlar = [], []
        for i in range(n):
            ad = '%s_%05d.jpg' % (bolum, i)
            im = Image.new('RGB', (320, 400), (random.randint(150, 255),) * 3)
            dr = ImageDraw.Draw(im)
            for k in range(3):
                x, y = random.randint(0, 200), random.randint(0, 260)
                w, h = random.randint(50, 120), random.randint(50, 140)
                c = random.choice(giysi)
                renk = tuple((c * 37 + j * 91) % 256 for j in range(3))   # tür → sabit renk: sahte veride öğrenilebilir
                cokgen = [x, y, x + w, y, x + w, y + h, x, y + h]
                dr.polygon(cokgen, fill=renk)
                a = random.sample(range(len(oz)), 3)
                notlar.append({'id': len(notlar) + 1, 'image_id': i + 1, 'category_id': c, 'attribute_ids': a,
                               'bbox': [x, y, w, h], 'area': w * h, 'iscrowd': 0, 'segmentation': [cokgen]})
                if c < 5:   # üst giyimde bir kol parçası
                    kw, kh = max(10, w // 4), max(10, h // 2)
                    notlar.append({'id': len(notlar) + 1, 'image_id': i + 1, 'category_id': kol_kat,
                                   'attribute_ids': [random.choice([13, 14, 15, 16])], 'bbox': [x, y, kw, kh], 'area': kw * kh,
                                   'iscrowd': 0, 'segmentation': [[x, y, x + kw, y, x + kw, y + kh, x, y + kh]]})
            im.save(os.path.join(VERI, klasor, ad), quality=85)
            resimler.append({'id': i + 1, 'file_name': ad, 'width': 320, 'height': 400, 'license': 1 + i % 3})
        js = {'info': {}, 'categories': [{'id': i, 'name': k, 'supercategory': 'x', 'level': 1} for i, k in enumerate(FP_KATEGORI)],
              'attributes': [{'id': i, 'name': a, 'supercategory': s, 'level': 1} for i, (a, s) in enumerate(oz)],
              'licenses': [{'id': 1, 'name': 'Attribution License', 'url': 'http://creativecommons.org/licenses/by/2.0/'},
                           {'id': 2, 'name': 'Attribution-NonCommercial License', 'url': 'http://creativecommons.org/licenses/by-nc/2.0/'},
                           {'id': 3, 'name': 'No known copyright restrictions', 'url': 'http://flickr.com/commons/usage/'}],
              'images': resimler, 'annotations': notlar}
        json.dump(js, open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum), 'w'))


if SINAMA:
    sahte_veri_kur()
else:
    for ad, adres in ADRESLER.items():
        indir(ad, adres)
    ac(os.path.join(VERI, 'train2020.zip'), os.path.join(VERI, 'train'))
    ac(os.path.join(VERI, 'val_test2020.zip'), os.path.join(VERI, 'test'))


def resim_klasoru(bolum):
    """Zip'ten çıkan klasör adını bulur (train / test)."""
    for aday in (['train', 'train2020'] if bolum == 'train2020' else ['test', 'val2020', 'val_test2020']):
        yol = os.path.join(VERI, aday)
        if os.path.isdir(yol):
            return yol
    raise FileNotFoundError('Resim klasörü bulunamadı: ' + bolum)


def lisans_uygun(lisans):
    """Ticari kullanıma kapalı (NonCommercial) fotoğraflar eğitime alınmaz."""
    if not lisans:
        return False
    metin = (lisans.get('name', '') + ' ' + lisans.get('url', '')).lower()
    return not re.search(r'noncommercial|non-commercial|by-nc|/nc', metin)


print({ad: os.path.getsize(os.path.join(VERI, ad)) // 1_000_000 for ad in os.listdir(VERI) if os.path.isfile(os.path.join(VERI, ad))}, 'MB')

## 2. Türleri Kombinle türlerine eşle
Her model sınıfı, uygulamadaki bir ya da birkaç türe karşılık gelir. Örneğin "gomlek_bluz" sınıfı
uygulamada önce *Bluz*, sonra *Gömlek* olarak önerilir. Kol boyu, giysinin üzerindeki "kol" parçalarının
etiketinden; etek/elbise boyu giysinin kendi etiketinden alınır. Bilgisi olmayan örnek o konuda eğitime katılmaz.

In [ ]:
# model sınıfı: (uygulamadaki tür adayları — ilk sıradaki öne çıkar, uygulamadaki grup)
SINIFLAR = {
    'tisort': (['tisort', 'bluz'], 'ust'),
    'sweat': (['sweat', 'kazak'], 'ust'),
    'askili': (['askili', 'bluz'], 'ust'),
    'gomlek_bluz': (['bluz', 'gomlek', 'tunik'], 'ust'),
    'tunik': (['tunik', 'bluz'], 'ust'),
    'kazak': (['kazak', 'sweat'], 'ust'),
    'hirka': (['hirka', 'kazak'], 'dis'),
    'blazer': (['blazer', 'ceket'], 'dis'),
    'ceket': (['ceket', 'blazer', 'mont'], 'dis'),
    'yelek': (['yelek'], 'dis'),
    'kaban': (['kaban', 'tren', 'mont'], 'dis'),
    'tren': (['tren', 'kaban'], 'dis'),
    'yagmurluk': (['yagmurluk', 'tren', 'mont'], 'dis'),
    'mont': (['mont', 'ceket'], 'dis'),
    'kot': (['kot'], 'alt'),
    'tayt': (['tayt', 'kumas'], 'alt'),
    'esofman': (['esofman'], 'alt'),
    'kumas': (['kumas', 'kot'], 'alt'),
    'sort': (['sort'], 'alt'),
    'etek': (['etek', 'uzunetek'], 'alt'),
    'uzunetek': (['uzunetek', 'etek'], 'alt'),
    'elbise': (['elbise', 'uzunelbise', 'tunik'], 'tek'),
    'uzunelbise': (['uzunelbise', 'elbise', 'ferace'], 'tek'),
    'tulum': (['tulum'], 'tek'),
    'ayakkabi': ([], 'ayk'),             # alt türünü uygulama kendi analiziyle seçer
    'bot': (['bot'], 'ayk'),
    'sandalet': (['sandalet', 'terlik'], 'ayk'),
    'topuklu': (['topuklu'], 'ayk'),
    'canta': (['canta'], 'aks'),
    'atki': (['atki', 'esarp'], 'aks'),
    'bere': (['bere'], 'aks'),
    'gozluk': (['gozluk'], 'aks'),
    'eldiven': (['eldiven'], 'aks'),
    'saat': (['saat'], 'aks'),
    'kemer': (['kemer'], 'aks'),
    'kupe': (['kupe'], 'aks'),
    'kolye': (['kolye'], 'aks'),
    'bileklik': (['bileklik', 'taki'], 'aks'),
    'kulotlu': (['kulotlu'], 'ic'),
    'corap': (['corap'], 'ic'),
    'sutyen': (['sutyen'], 'ic'),
    'yok': ([], ''),                     # kıyafet değil
}
DESENLER = ['duz', 'cizgili', 'kareli', 'cicekli', 'puantiyeli', 'hayvan', 'diger']
KOLLAR = ['kolsuz', 'kisa', 'orta', 'uzun']
BOYLAR = ['mini', 'dizustu', 'diz', 'midi', 'maksi']
DESEN_KURAL = [
    ('cizgili', r'stripe|chevron|herringbone|pinstripe'),
    ('kareli', r'check|plaid|tartan|houndstooth|argyle|gingham|gingam'),
    ('cicekli', r'floral|flower|plant|paisley|toile'),
    ('puantiyeli', r'\bdot|polka'),
    ('hayvan', r'leopard|cheetah|zebra|giraffe|snake|python|croc|animal|tiger'),
]
KOL_ALANI = ('shirt', 'top', 'sweater', 'cardigan', 'jacket', 'vest', 'coat', 'dress', 'jumpsuit')
BOY_ALANI = ('skirt', 'dress', 'coat')


def kombinle_sinifi(kat, oz):
    """Fashionpedia kategori adı + özellik adları → model sınıfı (ya da None)."""
    k = kat.lower()
    o = ' | '.join(sorted(oz))
    var = lambda desen: re.search(desen, o) is not None
    uzun = var(r'maxi|floor')
    if k.startswith('shirt'):
        return 'tunik' if var(r'tunic') else 'gomlek_bluz'
    if k.startswith('top'):
        if var(r'sweatshirt|hoodie'):
            return 'sweat'
        if var(r'tunic'):
            return 'tunik'
        if var(r'tank|camisole|halter|tube \(top\)|sleeveless'):
            return 'askili'
        return 'tisort'
    if k.startswith('sweater'):
        return 'kazak'
    if k.startswith('cardigan'):
        return 'hirka'
    if k.startswith('jacket'):
        if var(r'blazer'):
            return 'blazer'
        if var(r'rain|anorak'):
            return 'yagmurluk'
        if var(r'puffer|parka|down \(|quilted'):
            return 'mont'
        return 'ceket'
    if k.startswith('vest'):
        return 'yelek'
    if k.startswith('pants'):
        if var(r'jeans|denim'):
            return 'kot'
        if var(r'legging'):
            return 'tayt'
        if var(r'track|jogger|sweatpant'):
            return 'esofman'
        return 'kumas'
    if k.startswith('shorts'):
        return 'sort'
    if k.startswith('skirt'):
        return 'uzunetek' if uzun else 'etek'
    if k.startswith('coat'):
        if var(r'trench'):
            return 'tren'
        if var(r'rain|anorak'):
            return 'yagmurluk'
        if var(r'puffer|parka|down \('):
            return 'mont'
        return 'kaban'
    if k.startswith('dress'):
        return 'uzunelbise' if uzun else 'elbise'
    if k.startswith('jumpsuit'):
        return 'tulum'
    tablo = {'glasses': 'gozluk', 'hat': 'bere', 'glove': 'eldiven', 'watch': 'saat', 'belt': 'kemer',
             'tights': 'kulotlu', 'sock': 'corap', 'shoe': 'ayakkabi', 'bag': 'canta', 'scarf': 'atki'}
    for bas, sinif in tablo.items():
        if k.startswith(bas):
            return sinif
    return None


def desen_sinifi(desen_ozleri):
    """Desen özellikleri → desen sınıfı; bilgi yoksa -1 (eğitimde sayılmaz)."""
    if not desen_ozleri:
        return -1
    o = ' | '.join(sorted(x.lower() for x in desen_ozleri))
    for ad, kural in DESEN_KURAL:
        if re.search(kural, o):
            return DESENLER.index(ad)
    if re.search(r'plain', o) and len(desen_ozleri) == 1:
        return DESENLER.index('duz')
    return DESENLER.index('diger')


def kol_sinifi(adlar, parca=True):
    """Kol parçasının (ya da giysinin) boy etiketi → kol sınıfı; yoksa -1."""
    for a in adlar:
        if 'sleeveless' in a:
            return KOLLAR.index('kolsuz')
        if not parca:
            continue
        if a.startswith('wrist'):
            return KOLLAR.index('uzun')
        if a.startswith('elbow') or a.startswith('three quarter') or a.startswith('three-quarter'):
            return KOLLAR.index('orta')
        if a.startswith('short (length)') or a.startswith('short'):
            return KOLLAR.index('kisa')
    return -1


def boy_sinifi(adlar):
    for a in adlar:
        if a.startswith('micro') or a.startswith('mini'):
            return BOYLAR.index('mini')
        if a.startswith('above-the-knee') or a.startswith('above the knee'):
            return BOYLAR.index('dizustu')
        if a.startswith('knee'):
            return BOYLAR.index('diz')
        if a.startswith('below the knee') or a.startswith('below-the-knee') or a.startswith('midi'):
            return BOYLAR.index('midi')
        if a.startswith('maxi') or a.startswith('floor'):
            return BOYLAR.index('maksi')
    return -1


def ic_orani(ic, dis):
    """ic kutusunun yüzde kaçı dis kutusunun içinde ([x, y, w, h])."""
    x0, y0 = max(ic[0], dis[0]), max(ic[1], dis[1])
    x1, y1 = min(ic[0] + ic[2], dis[0] + dis[2]), min(ic[1] + ic[3], dis[1] + dis[3])
    return max(0, x1 - x0) * max(0, y1 - y0) / max(1e-6, ic[2] * ic[3])


def ornekleri_cikar(bolum, sinir=None):
    js = json.load(open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum)))
    kat = {c['id']: c['name'] for c in js['categories']}
    oz_ad = {a['id']: a['name'].lower() for a in js['attributes']}
    oz_ust = {a['id']: (a.get('supercategory') or '').lower() for a in js['attributes']}
    lis = {l['id']: l for l in js.get('licenses', [])}
    resim = {r['id']: r for r in js['images']}
    if not hasattr(ornekleri_cikar, 'yazildi'):
        ornekleri_cikar.yazildi = True
        print('Desen özellikleri:', sorted(oz_ad[i] for i in oz_ad if 'pattern' in oz_ust[i])[:40])
        print('Boy özellikleri:', sorted(oz_ad[i] for i in oz_ad if 'length' in oz_ust[i] or 'length' in oz_ad[i])[:40])
        print('Lisanslar:', [(l['id'], l.get('name')) for l in js.get('licenses', [])])
    resim_notlari = defaultdict(list)
    for n in js['annotations']:
        resim_notlari[n['image_id']].append(n)
    lisans_say, ornekler, kol_say = Counter(), [], Counter()
    for rid, notlar in resim_notlari.items():
        r = resim.get(rid)
        if not r:
            continue
        uygun = lisans_uygun(lis.get(r.get('license')))
        lisans_say['uygun' if uygun else 'ticari değil / bilinmiyor'] += len(notlar)
        if not uygun:
            continue
        # kol parçalarını içinde bulundukları giysiye bağla
        kollar = [n for n in notlar if kat.get(n['category_id'], '').lower().startswith('sleeve')]
        adaylar = [n for n in notlar if kat.get(n['category_id'], '').lower().startswith(KOL_ALANI)]
        oy = defaultdict(Counter)
        for k in kollar:
            ks = kol_sinifi([oz_ad[a] for a in k.get('attribute_ids', []) if a in oz_ad])
            if ks < 0 or not adaylar:
                continue
            en = max(adaylar, key=lambda g: (round(ic_orani(k['bbox'], g['bbox']), 2), g['bbox'][2] * g['bbox'][3]))
            if ic_orani(k['bbox'], en['bbox']) >= 0.6:
                oy[en['id']][ks] += 1
        for n in notlar:
            x, y, w, h = n['bbox']
            if w < EN_KUCUK_KUTU or h < EN_KUCUK_KUTU:
                continue
            kat_ad = kat.get(n['category_id'], '').lower()
            ozler = [oz_ad[a] for a in n.get('attribute_ids', []) if a in oz_ad]
            sinif = kombinle_sinifi(kat_ad, ozler)
            if not sinif:
                continue
            desen = desen_sinifi([oz_ad[a] for a in n.get('attribute_ids', []) if a in oz_ad and 'pattern' in oz_ust[a]])
            kol = -1
            if kat_ad.startswith(KOL_ALANI):
                kol = kol_sinifi(ozler, parca=False)
                if kol < 0 and oy[n['id']]:
                    kol = oy[n['id']].most_common(1)[0][0]
            boy = boy_sinifi(ozler) if kat_ad.startswith(BOY_ALANI) else -1
            kol_say[KOLLAR[kol] if kol >= 0 else '?'] += 1
            ornekler.append({'resim': r['file_name'], 'kutu': [x, y, w, h], 'sinif': sinif, 'desen': desen, 'kol': kol, 'boy': boy,
                             'kaynak': 'fp'})
    print(bolum, 'lisans:', dict(lisans_say), '| kol:', dict(kol_say))
    if sinir:
        gruplu = defaultdict(list)
        for o in ornekler:
            gruplu[o['sinif']].append(o)
        ornekler = []
        for s, l in gruplu.items():
            random.shuffle(l)
            ornekler += l[:sinir]
    return ornekler


fp_egitim = ornekleri_cikar('train2020', SINIF_BASI_EN_FAZLA)
fp_dogrulama = ornekleri_cikar('val2020')
print('Fashionpedia örnekleri:', len(fp_egitim), '/', len(fp_dogrulama))

## 3. Open Images'tan ek türler
Fashionpedia'da olmayanlar Open Images'tan gelir: **bot, sandalet, topuklu, küpe, kolye, bileklik, sütyen**,
aksesuarlara ek örnekler ve **"kıyafet değil"** örnekleri (yemek, araba, hayvan, eşya…).
Etiket dosyası ~2 GB; yalnız gereken fotoğraflar tek tek indirilir, kırpılır ve aslı silinir (20–40 dakika).
Open Images'a ulaşılamazsa bu adım atlanır, model yalnız Fashionpedia ile eğitilir.

In [ ]:
from PIL import Image
import io, urllib.request
from concurrent.futures import ThreadPoolExecutor

OI_ADRES = {
    'siniflar': 'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv',
    'egitim': 'https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv',
    'dogrulama': 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv',
}
OI_RESIM = ['https://open-images-dataset.s3.amazonaws.com/{bolum}/{id}.jpg',
            'https://s3.amazonaws.com/open-images-dataset/{bolum}/{id}.jpg']
# Kombinle sınıfı → Open Images sınıf adları (adı veride yoksa atlanır)
OI_ESLE = {
    'bot': ['Boot'], 'sandalet': ['Sandal'], 'topuklu': ['High heels'], 'ayakkabi': ['Footwear'],
    'kupe': ['Earrings'], 'kolye': ['Necklace'], 'bileklik': ['Bracelet', 'Bangle'], 'sutyen': ['Brassiere'],
    'canta': ['Handbag', 'Backpack'], 'gozluk': ['Sunglasses', 'Glasses'], 'bere': ['Hat', 'Sun hat', 'Fedora', 'Cowboy hat'],
    'saat': ['Watch'], 'eldiven': ['Glove'], 'corap': ['Sock'], 'atki': ['Scarf'], 'kemer': ['Belt'],
}
AYAKKABI_ALT = ('bot', 'sandalet', 'topuklu')
# kıyafet değil: içinde giysi olması pek beklenmeyen nesneler
YOK_ADLARI = ['Food', 'Fruit', 'Vegetable', 'Baked goods', 'Dessert', 'Pizza', 'Cake', 'Car', 'Bus', 'Truck', 'Bicycle', 'Motorcycle',
              'Boat', 'Airplane', 'Dog', 'Cat', 'Bird', 'Horse', 'Fish', 'Houseplant', 'Flower', 'Tree', 'Building', 'House',
              'Laptop', 'Mobile phone', 'Computer keyboard', 'Computer monitor', 'Television', 'Book', 'Bottle', 'Mug', 'Coffee cup',
              'Wine glass', 'Bowl', 'Plate', 'Lamp', 'Clock', 'Toy', 'Teddy bear', 'Ball', 'Kitchen appliance', 'Sink', 'Toilet',
              'Bed', 'Chair', 'Table', 'Window', 'Door', 'Drink', 'Candle', 'Vase', 'Pillow']
OI_MIN = {'yok': 0.25}             # kutunun en kısa kenarı, fotoğrafın en az bu kadarı
OI_MIN_VARSAYILAN = 0.06


def oi_csv_oku(yol, mid_sinif, yok_mid):
    """Gereken satırları akış hâlinde süzer. Dönüş: {sinif: [(resim, x0, y0, x1, y1)]}, ayakkabı alt türü kutuları."""
    kutular, alt_kutu = defaultdict(list), defaultdict(list)
    with open(yol, newline='') as f:
        r = csv.reader(f)
        bas = next(r)
        i = {k: n for n, k in enumerate(bas)}
        for s in r:
            mid = s[i['LabelName']]
            sinif = mid_sinif.get(mid) or ('yok' if mid in yok_mid else None)
            if not sinif or s[i['IsGroupOf']] == '1' or s[i['IsDepiction']] == '1':
                continue
            x0, x1, y0, y1 = float(s[i['XMin']]), float(s[i['XMax']]), float(s[i['YMin']]), float(s[i['YMax']])
            if sinif in AYAKKABI_ALT:
                alt_kutu[s[i['ImageID']]].append((x0, y0, x1, y1))
            if min(x1 - x0, y1 - y0) < OI_MIN.get(sinif, OI_MIN_VARSAYILAN):
                continue
            kutular[sinif].append((s[i['ImageID']], x0, y0, x1, y1, yok_mid.get(mid, '')))
    return kutular, alt_kutu


def kesisim_birlesim(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    k = max(0, x1 - x0) * max(0, y1 - y0)
    return k / max(1e-9, (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - k)


def oi_sec(kutular, alt_kutu, sinif_basi, yok_toplam):
    secilen = []
    for sinif, l in kutular.items():
        if sinif == 'ayakkabi':   # genel "ayakkabı": aynı yerde bot/sandalet/topuklu kutusu olmayanlar
            l = [k for k in l if not any(kesisim_birlesim(k[1:5], a) > 0.3 for a in alt_kutu.get(k[0], []))]
        random.shuffle(l)
        if sinif == 'yok':   # her nesne türünden eşit pay
            gruplu = defaultdict(list)
            for k in l:
                gruplu[k[5]].append(k)
            pay = max(1, yok_toplam // max(1, len(gruplu)))
            l = [k for g in gruplu.values() for k in g[:pay]][:yok_toplam]
        else:
            l = l[:sinif_basi]
        secilen += [{'resim': k[0], 'kutu': k[1:5], 'sinif': sinif, 'desen': -1, 'kol': -1, 'boy': -1, 'kaynak': 'oi'} for k in l]
    return secilen


def oi_resim_getir(bolum, rid):
    if SINAMA:   # sahte fotoğraf
        from PIL import ImageDraw
        im = Image.new('RGB', (400, 300), (random.randint(100, 255),) * 3)
        ImageDraw.Draw(im).ellipse([60, 40, 340, 260], fill=tuple(random.randint(0, 255) for _ in range(3)))
        return im
    for adres in OI_RESIM:
        try:
            with urllib.request.urlopen(adres.format(bolum=bolum, id=rid), timeout=40) as y:
                return Image.open(io.BytesIO(y.read())).convert('RGB')
        except Exception:
            continue
    return None


def oi_isle(is_):
    (bolum, rid), gorevler = is_
    im = oi_resim_getir(bolum, rid)
    if im is None:
        return 0
    W, H = im.size
    n = 0
    for hedef, (x0, y0, x1, y1) in gorevler:
        x0, x1, y0, y1 = x0 * W, x1 * W, y0 * H, y1 * H
        p = 0.08 * max(x1 - x0, y1 - y0)
        kutu = (max(0, x0 - p), max(0, y0 - p), min(W, x1 + p), min(H, y1 + p))
        kareye_tamamla(im.crop(tuple(int(round(v)) for v in kutu)), KAYIT_BOYU).save(hedef, quality=90)
        n += 1
    return n


def kareye_tamamla(im, boy):
    im = im.convert('RGB')
    w, h = im.size
    o = boy / max(w, h)
    yeni = im.resize((max(1, round(w * o)), max(1, round(h * o))), Image.BILINEAR)
    zemin = Image.new('RGB', (boy, boy), (255, 255, 255))
    zemin.paste(yeni, ((boy - yeni.width) // 2, (boy - yeni.height) // 2))
    return zemin


def oi_kirp(ornekler, bolum, ad):
    hedef_klasor = os.path.join(KIRPIK, 'oi_' + ad)
    os.makedirs(hedef_klasor, exist_ok=True)
    isler = defaultdict(list)
    for i, o in enumerate(ornekler):
        o['dosya'] = os.path.join(hedef_klasor, '%s_%s_%d.jpg' % (o['resim'], o['sinif'], i))
        if not os.path.exists(o['dosya']):
            isler[(bolum, o['resim'])].append((o['dosya'], o['kutu']))
    bas = time.time()
    isler = list(isler.items())
    # önce birkaç fotoğrafla dene: Open Images'a ulaşılamıyorsa saatlerce beklemeden vazgeç
    deneme = [oi_isle(i) for i in isler[:8]]
    if isler[:8] and not any(deneme):
        raise RuntimeError('Open Images fotoğrafları indirilemiyor')
    with ThreadPoolExecutor(max_workers=48) as h:
        for k, _ in enumerate(h.map(oi_isle, isler[8:])):
            if k and k % 2000 == 0:
                print('  %d / %d fotoğraf (%.0f sn)' % (k, len(isler), time.time() - bas))
    tamam = [o for o in ornekler if os.path.exists(o['dosya'])]
    print('Open Images', ad, ':', len(tamam), 'örnek hazır (%d fotoğraf indirilemedi)' % (len(ornekler) - len(tamam)))
    return tamam


def sahte_oi_kur():
    adlar = sorted({a for l in OI_ESLE.values() for a in l} | set(YOK_ADLARI[:6]))
    mid = {a: '/m/%04d' % i for i, a in enumerate(adlar)}
    with open(os.path.join(OI, 'siniflar.csv'), 'w') as f:
        for a in adlar:
            f.write('%s,%s\n' % (mid[a], a))
    for ad, n in (('egitim', 900), ('dogrulama', 200)):
        with open(os.path.join(OI, ad + '.csv'), 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['ImageID', 'Source', 'LabelName', 'Confidence', 'XMin', 'XMax', 'YMin', 'YMax', 'IsOccluded', 'IsTruncated',
                        'IsGroupOf', 'IsDepiction', 'IsInside'])
            for i in range(n):
                a = random.choice(adlar)
                w.writerow(['%s%05d' % (ad, i), 'x', mid[a], 1, 0.1, 0.8, 0.15, 0.9, 0, 0, 0, 0, 0])


oi_egitim, oi_dogrulama = [], []
try:
    if SINAMA:
        sahte_oi_kur()
        yollar = {k: os.path.join(OI, k + '.csv') for k in ('siniflar', 'egitim', 'dogrulama')}
    else:
        yollar = {k: indir(k + '.csv', v, OI) for k, v in OI_ADRES.items()}
    ad_mid = {}
    with open(yollar['siniflar'], newline='') as f:
        for s in csv.reader(f):
            if len(s) >= 2:
                ad_mid[s[1].strip()] = s[0].strip()
    mid_sinif = {ad_mid[a]: s for s, l in OI_ESLE.items() for a in l if a in ad_mid}
    yok_mid = {ad_mid[a]: a for a in YOK_ADLARI if a in ad_mid}
    eksik = [a for l in OI_ESLE.values() for a in l if a not in ad_mid] + [a for a in YOK_ADLARI if a not in ad_mid]
    print('Open Images: %d kıyafet sınıfı, %d "kıyafet değil" nesnesi eşlendi; veride olmayan: %s' % (len(mid_sinif), len(yok_mid), eksik or 'yok'))
    k_e, a_e = oi_csv_oku(yollar['egitim'], mid_sinif, yok_mid)
    k_d, a_d = oi_csv_oku(yollar['dogrulama'], mid_sinif, yok_mid)
    print('Uygun kutular:', {s: len(l) for s, l in sorted(k_e.items())})
    oi_egitim = oi_kirp(oi_sec(k_e, a_e, OI_SINIF_BASI, OI_YOK_TOPLAM), 'train', 'egitim')
    oi_dogrulama = oi_kirp(oi_sec(k_d, a_d, OI_DOGRULAMA_BASI, OI_DOGRULAMA_BASI * 4), 'validation', 'dogrulama')
except Exception as e:
    print('⚠️  Open Images adımı atlandı (%s). Model yalnız Fashionpedia ile eğitilecek.' % e)
    oi_egitim, oi_dogrulama = [], []

## 4. Türleri birleştir, kıyafetleri kırp
Her kıyafet kutusu biraz boşlukla kesilir ve beyaz zeminle kareye tamamlanır.
Uygulama da telefonda fotoğrafı aynı şekilde kareye tamamlıyor.

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

# Ayakkabı: Open Images'ta bot / sandalet / topuklu ayrı ayrı yeterliyse Fashionpedia'nın genel "ayakkabı" örnekleri çıkarılır
# (aynı bota hem "ayakkabı" hem "bot" denmesin). Değilse alt türler çıkarılır, genel "ayakkabı" kalır.
oi_say = Counter(o['sinif'] for o in oi_egitim)
alt_tamam = all(oi_say[s] >= SINIF_BASI_EN_AZ * (1 if not SINAMA else 0.02) for s in AYAKKABI_ALT)
if alt_tamam:
    fp_egitim = [o for o in fp_egitim if o['sinif'] != 'ayakkabi']
    fp_dogrulama = [o for o in fp_dogrulama if o['sinif'] != 'ayakkabi']
    print('Ayakkabı alt türleri (bot, sandalet, topuklu) modelde.')
else:
    oi_egitim = [o for o in oi_egitim if o['sinif'] not in AYAKKABI_ALT + ('ayakkabi',)]
    oi_dogrulama = [o for o in oi_dogrulama if o['sinif'] not in AYAKKABI_ALT + ('ayakkabi',)]
    print('Ayakkabı alt türleri için yeterli veri yok; model genel "ayakkabı" der.')

egitim_ornek = fp_egitim + oi_egitim
dogrulama_ornek = fp_dogrulama + oi_dogrulama
sayim = Counter(o['sinif'] for o in egitim_ornek)
SINIF_LISTESI = [s for s in SINIFLAR if sayim[s] >= (SINIF_BASI_EN_AZ if not SINAMA else 3)]
dusen = [s for s in SINIFLAR if s not in SINIF_LISTESI]
egitim_ornek = [o for o in egitim_ornek if o['sinif'] in SINIF_LISTESI]
dogrulama_ornek = [o for o in dogrulama_ornek if o['sinif'] in SINIF_LISTESI]
print('\nModele giren türler (%d):' % len(SINIF_LISTESI))
for s in SINIF_LISTESI:
    print('  %-12s %5d' % (s, sayim[s]))
print('Yeterli örneği olmadığı için çıkan:', dusen or 'yok')

# kol / boy bilgisi yeterli değilse o çıkış hiç eklenmez
BASLAR = {'tur': SINIF_LISTESI, 'desen': DESENLER, 'kol': KOLLAR, 'boy': BOYLAR}
for bas in ('kol', 'boy'):
    n = sum(o[bas] >= 0 for o in egitim_ornek)
    print('%s bilgisi olan örnek: %d' % (bas, n), Counter(BASLAR[bas][o[bas]] for o in egitim_ornek if o[bas] >= 0))
    if n < (500 if not SINAMA else 5):
        print('  → yeterli değil, "%s" çıkışı eklenmiyor' % bas)
        del BASLAR[bas]
boylar = [len(v) for v in BASLAR.values()]
assert len(set(boylar)) == len(boylar), 'çıkışlar boylarından ayırt ediliyor; aynı boyda iki çıkış olamaz: %s' % boylar
print('Desen dağılımı:', Counter(DESENLER[o['desen']] if o['desen'] >= 0 else 'bilinmiyor' for o in egitim_ornek))


def resmi_isle(is_):
    yol, gorevler = is_
    cikan = []
    try:
        with Image.open(yol) as im:
            im.load()
            W, H = im.size
            for hedef, (x, y, w, h) in gorevler:
                p = 0.08 * max(w, h)
                kutu = (max(0, x - p), max(0, y - p), min(W, x + w + p), min(H, y + h + p))
                kareye_tamamla(im.crop(tuple(int(round(v)) for v in kutu)), KAYIT_BOYU).save(hedef, quality=90)
                cikan.append(hedef)
    except Exception as e:
        print('atlandı:', os.path.basename(yol), e)
    return cikan


def kirp(ornekler, bolum):
    klasor = resim_klasoru(bolum)
    hedef_klasor = os.path.join(KIRPIK, bolum)
    os.makedirs(hedef_klasor, exist_ok=True)
    isler = defaultdict(list)
    for i, o in enumerate(ornekler):
        o['dosya'] = os.path.join(hedef_klasor, '%06d_%s.jpg' % (i, o['sinif']))
        if not os.path.exists(o['dosya']):
            isler[os.path.join(klasor, o['resim'])].append((o['dosya'], o['kutu']))
    bas = time.time()
    with ProcessPoolExecutor(max_workers=max(2, os.cpu_count() or 2), mp_context=multiprocessing.get_context('fork')) as h:
        for k, _ in enumerate(h.map(resmi_isle, list(isler.items()), chunksize=32)):
            if k and k % 2000 == 0:
                print('  %d / %d fotoğraf (%.0f sn)' % (k, len(isler), time.time() - bas))
    tamam = [o for o in ornekler if os.path.exists(o['dosya'])]
    print(bolum, ':', len(tamam), 'örnek hazır')
    return tamam


egitim_ornek = kirp([o for o in egitim_ornek if o['kaynak'] == 'fp'], 'train2020') + [o for o in egitim_ornek if o['kaynak'] == 'oi']
dogrulama_ornek = kirp([o for o in dogrulama_ornek if o['kaynak'] == 'fp'], 'val2020') + [o for o in dogrulama_ornek if o['kaynak'] == 'oi']
random.shuffle(egitim_ornek)
print('Eğitim:', len(egitim_ornek), '| Doğrulama:', len(dogrulama_ornek))

## 5. Modeli eğit
Temel: ImageNet'te eğitilmiş **EfficientNetV2-B2**. Üstüne dört çıkış eklenir: tür, desen, kol boyu, etek/elbise boyu.
Önce yalnız yeni katmanlar, sonra modelin üst kısmı ince ayarlanır. Her dönem sonunda Drive'a kaydedilir.

In [ ]:
if tf is None:
    raise SystemExit('TensorFlow gerekli (Colab\'da hazır gelir).')
import numpy as np
from keras import layers, ops

keras.utils.set_random_seed(TOHUM)
SINIF_NO = {s: i for i, s in enumerate(SINIF_LISTESI)}
ETIKET = {'tur': lambda o: SINIF_NO[o['sinif']], 'desen': lambda o: o['desen'], 'kol': lambda o: o['kol'], 'boy': lambda o: o['boy']}


def veri_kumesi(ornekler, egitim, toplu=TOPLU):
    yollar = [o['dosya'] for o in ornekler]
    etiket = {b: np.array([ETIKET[b](o) for o in ornekler], dtype=np.int32) for b in BASLAR}
    ds = tf.data.Dataset.from_tensor_slices((yollar, etiket))
    if egitim:
        ds = ds.shuffle(len(yollar), seed=TOHUM, reshuffle_each_iteration=True)

    def yukle(yol, e):
        im = tf.io.decode_jpeg(tf.io.read_file(yol), channels=3)
        im = tf.image.convert_image_dtype(im, tf.float32)       # 0..1
        if egitim:
            olcek = tf.random.uniform([], 0.8, 1.0)
            kenar = tf.cast(tf.cast(KAYIT_BOYU, tf.float32) * olcek, tf.int32)
            im = tf.image.random_crop(im, [kenar, kenar, 3])
            im = tf.image.random_flip_left_right(im)
            im = tf.image.random_brightness(im, 0.12)
            im = tf.image.random_contrast(im, 0.85, 1.15)
            im = tf.image.random_saturation(im, 0.85, 1.15)
            im = tf.clip_by_value(im, 0.0, 1.0)
        im = tf.image.resize(im, [GIRDI, GIRDI], antialias=True) * 255.0   # model 0..255 bekler
        e = dict(e)
        e['tur'] = tf.one_hot(e['tur'], len(SINIF_LISTESI))
        return im, e

    return ds.map(yukle, num_parallel_calls=tf.data.AUTOTUNE).batch(toplu).prefetch(tf.data.AUTOTUNE)


def maskeli_kayip(y_dogru, y_tahmin):
    """Bilgisi olmayan örnekler (-1) kayba katılmaz."""
    y = ops.reshape(ops.cast(y_dogru, 'int32'), (-1,))
    gecerli = ops.cast(y >= 0, 'float32')
    k = keras.losses.sparse_categorical_crossentropy(ops.maximum(y, 0), ops.cast(y_tahmin, 'float32'))
    return ops.sum(k * gecerli) / ops.maximum(ops.sum(gecerli), 1.0)


def model_kur(agirlik='imagenet'):
    Taban = getattr(keras.applications, TABAN)
    ek = {'include_preprocessing': True}
    taban = Taban(input_shape=(GIRDI, GIRDI, 3), include_top=False, weights=agirlik, pooling='avg', **ek)
    taban.trainable = False
    giris = keras.Input((GIRDI, GIRDI, 3), name='resim')
    x = taban(giris, training=False)
    x = layers.Dropout(0.3)(x)
    cikis = {'tur': layers.Dense(len(SINIF_LISTESI), activation='softmax', name='tur', dtype='float32')(x)}
    ara = layers.Dropout(0.2)(x)
    for bas in ('desen', 'kol', 'boy'):
        if bas in BASLAR:
            cikis[bas] = layers.Dense(len(BASLAR[bas]), activation='softmax', name=bas, dtype='float32')(ara)
    return keras.Model(giris, cikis), taban


egitim_ds = veri_kumesi(egitim_ornek, True)
dogrulama_ds = veri_kumesi(dogrulama_ornek, False)
model, taban = model_kur()

DURUM = os.path.join(KALICI, 'tur_durum.json')
AGIRLIK = os.path.join(KALICI, 'tur_son.weights.h5')
EN_IYI = os.path.join(KALICI, 'tur_en_iyi.weights.h5')
IMZA = {'siniflar': SINIF_LISTESI, 'baslar': list(BASLAR), 'taban': TABAN, 'girdi': GIRDI, 'sinama': SINAMA}
durum = {'asama': 0, 'donem': 0}
if os.path.exists(DURUM) and os.path.exists(AGIRLIK):
    try:
        d = json.load(open(DURUM))
        if d.get('imza') == IMZA:
            model.load_weights(AGIRLIK)
            durum = d
            print('Kaldığı yerden devam: %d. aşama, %d. dönem' % (durum['asama'], durum['donem']))
        else:
            print('Önceki kayıt farklı bir ayarla yapılmış; baştan başlanıyor.')
    except Exception as e:
        print('Önceki kayıt okunamadı, baştan başlanıyor:', e)


class DriveKaydi(keras.callbacks.Callback):
    def __init__(self, asama):
        super().__init__()
        self.asama = asama

    def on_epoch_end(self, donem, logs=None):
        self.model.save_weights(AGIRLIK)
        json.dump({'asama': self.asama, 'donem': donem + 1, 'imza': IMZA}, open(DURUM, 'w'))


def derle(oran):
    kayip = {'tur': keras.losses.CategoricalCrossentropy(label_smoothing=0.1)}
    agirlik = {'tur': 1.0}
    for bas, a in (('desen', 0.5), ('kol', 0.3), ('boy', 0.3)):
        if bas in BASLAR:
            kayip[bas], agirlik[bas] = maskeli_kayip, a
    model.compile(optimizer=keras.optimizers.Adam(oran), loss=kayip, loss_weights=agirlik, metrics={'tur': [keras.metrics.CategoricalAccuracy(name='dogruluk')]})


ISINMA = 1 if SINAMA else DONEM_ISINMA
INCE = 1 if SINAMA else DONEM_INCE
ayrinti = 2 if SINAMA else 1
if durum['asama'] <= 1 and not (durum['asama'] == 1 and durum['donem'] >= ISINMA):
    print('1. aşama: yeni katmanlar')
    derle(1e-3)
    model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=ISINMA, initial_epoch=durum['donem'] if durum['asama'] == 1 else 0,
              verbose=ayrinti, callbacks=[DriveKaydi(1)])
    durum = {'asama': 2, 'donem': 0}

print('2. aşama: ince ayar')
taban.trainable = True
for katman in taban.layers[:int(len(taban.layers) * 0.45)]:
    katman.trainable = False
for katman in taban.layers:
    if isinstance(katman, layers.BatchNormalization):
        katman.trainable = False
derle(1e-4)
basla = durum['donem'] if durum['asama'] == 2 else 0
if basla < INCE:
    model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=INCE, initial_epoch=basla, verbose=ayrinti,
              callbacks=[DriveKaydi(2),
                         keras.callbacks.ModelCheckpoint(EN_IYI, monitor='val_loss', save_best_only=True, save_weights_only=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
                         keras.callbacks.EarlyStopping(monitor='val_loss', patience=4)])
if os.path.exists(EN_IYI):
    model.load_weights(EN_IYI)
    print('En iyi dönemin ağırlıkları yüklendi.')

## 6. Ne kadar doğru?
Eğitimde hiç görmediği doğrulama fotoğraflarıyla ölçülür.

In [ ]:
tahmin = model.predict(dogrulama_ds, verbose=0)
dogru_tur = np.array([SINIF_NO[o['sinif']] for o in dogrulama_ornek])
tur_t = tahmin['tur'].argmax(1)
tur_dogruluk = float((tur_t == dogru_tur).mean())
emin = tahmin['tur'].max(1) >= 0.75
emin_dogruluk = float((tur_t[emin] == dogru_tur[emin]).mean()) if emin.any() else 0.0
dogruluk = {'tur': round(tur_dogruluk, 3), 'emin75': round(emin_dogruluk, 3), 'emin75_oran': round(float(emin.mean()), 3),
            'ornek': len(dogrulama_ornek)}


def bas_dogrulugu(bas):
    d = np.array([o[bas] for o in dogrulama_ornek])
    m = d >= 0
    return (float((tahmin[bas].argmax(1)[m] == d[m]).mean()) if m.any() else 0.0), int(m.sum())


print('Tür doğruluğu        : %%%.1f' % (tur_dogruluk * 100))
print('Model emin (≥%%75) olduğunda: %%%.1f doğru, örneklerin %%%.0f\'inde emin' % (emin_dogruluk * 100, emin.mean() * 100))
for bas, ad in (('desen', 'Desen türü'), ('kol', 'Kol boyu'), ('boy', 'Etek/elbise boyu')):
    if bas in BASLAR:
        v, n = bas_dogrulugu(bas)
        dogruluk[bas] = round(v, 3)
        print('%-21s: %%%.1f  (%d örnek)' % (ad, v * 100, n))
if 'desen' in BASLAR:
    d = np.array([o['desen'] for o in dogrulama_ornek]); m = d >= 0; duz = DESENLER.index('duz')
    dogruluk['duz_desenli'] = round(float(((tahmin['desen'].argmax(1)[m] == duz) == (d[m] == duz)).mean()) if m.any() else 0.0, 3)
    print('Düz / desenli ayrımı : %%%.1f' % (dogruluk['duz_desenli'] * 100))
YOK_ESIK = 0.9
if 'yok' in SINIF_NO:
    yi = SINIF_NO['yok']
    dedi = (tur_t == yi) & (tahmin['tur'][:, yi] >= YOK_ESIK)
    gercek = dogru_tur == yi
    dogruluk['yok_kesin'] = round(float(gercek[dedi].mean()) if dedi.any() else 0.0, 3)
    dogruluk['kiyafet_kacirma'] = round(float(dedi[~gercek].mean()) if (~gercek).any() else 0.0, 4)
    print('"Kıyafet değil" dediğinde haklı: %%%.1f | kıyafete yanlışlıkla "değil" deme: %%%.2f'
          % (dogruluk['yok_kesin'] * 100, dogruluk['kiyafet_kacirma'] * 100))
print('\nTür bazında:')
for s, i in SINIF_NO.items():
    sec = dogru_tur == i
    if sec.any():
        print('  %-12s %%%5.1f  (%d örnek)' % (s, (tur_t[sec] == i).mean() * 100, sec.sum()))

## 7. Telefon modeline dönüştür
Model küçültülür (8 bit ağırlık) ve telefondaki sonuçla buradaki sonuç karşılaştırılır.

In [ ]:
# telefona float32 model gider: aynı yapı float32 kurulup ağırlıklar aktarılır
keras.mixed_precision.set_global_policy('float32')
model32, _ = model_kur(agirlik=None)
model32.set_weights(model.get_weights())

kayit_yolu = os.path.join(CALISMA, 'kayitli_model')
shutil.rmtree(kayit_yolu, ignore_errors=True)
model32.export(kayit_yolu)
MODEL_YOLU = os.path.join(CIKTI, 'kombinle_model.tflite')
ornek_say = min(200, len(dogrulama_ornek))
toplu = next(iter(veri_kumesi(dogrulama_ornek[:ornek_say], False, ornek_say)))[0].numpy()
keras_t = model32.predict(toplu, verbose=0)['tur'].argmax(1)


def donustur(kip):
    d = tf.lite.TFLiteConverter.from_saved_model(kayit_yolu)
    d.optimizations = [tf.lite.Optimize.DEFAULT]
    if kip == '16 bit':
        d.target_spec.supported_types = [tf.float16]
    return d.convert()


# telefondaki gibi çalıştırıp karşılaştır: çıkışlar boylarına göre ayırt edilir.
# 8 bit küçültme sonucu fazla değiştirirse 16 bit (daha büyük, neredeyse kayıpsız) denenir.
for kip in ('8 bit', '16 bit'):
    tflite = donustur(kip)
    yorumlayici = tf.lite.Interpreter(model_content=tflite)
    yorumlayici.allocate_tensors()
    giris_d = yorumlayici.get_input_details()[0]
    cikis_d = yorumlayici.get_output_details()
    assert list(giris_d['shape']) == [1, GIRDI, GIRDI, 3] and giris_d['dtype'] == np.float32, giris_d
    boy = {int(c['shape'][-1]): c['index'] for c in cikis_d}
    assert all(len(v) in boy for v in BASLAR.values()), boy
    uyan = 0
    for i in range(ornek_say):
        yorumlayici.set_tensor(giris_d['index'], toplu[i:i + 1])
        yorumlayici.invoke()
        uyan += int(yorumlayici.get_tensor(boy[len(SINIF_LISTESI)])[0].argmax() == keras_t[i])
    print('%s: model boyutu %.1f MB, telefon modeli ile eğitim modeli aynı türü buldu: %d / %d' % (kip, len(tflite) / 1e6, uyan, ornek_say))
    if uyan >= ornek_say * 0.9:
        break
assert uyan >= ornek_say * 0.9, 'Dönüştürmede sorun var; modeli göndermeyin.'
open(MODEL_YOLU, 'wb').write(tflite)

bilgi = {
    'surum': int(time.time()),
    'bicim': 2,
    'taban': TABAN,
    'girdi': GIRDI,
    'olcek': '0-255',
    'zemin': '#ffffff',
    'siniflar': [{'id': s, 'turlar': SINIFLAR[s][0], 'grup': SINIFLAR[s][1]} for s in SINIF_LISTESI],
    'desenler': DESENLER,
    'cikislar': {b: len(v) for b, v in BASLAR.items()},
    'yok_esik': YOK_ESIK,
    'dogruluk': dogruluk,
    'kaynak': 'Fashionpedia (CC BY 4.0) + Open Images (CC BY 4.0 / CC BY 2.0)' if oi_egitim else 'Fashionpedia (etiketler CC BY 4.0)',
}
if 'kol' in BASLAR:
    bilgi['kollar'] = KOLLAR
if 'boy' in BASLAR:
    bilgi['boylar'] = BOYLAR
BILGI_YOLU = os.path.join(CIKTI, 'kombinle_model.json')
json.dump(bilgi, open(BILGI_YOLU, 'w'), ensure_ascii=False, indent=1)
print(json.dumps(bilgi['dogruluk']))

## Modeli Kombinle'ye gönder
Bir **GitHub anahtarı** gerekiyor (bir kez):
1. GitHub → sağ üst fotoğraf → **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**
2. *Repository access*: **Only select repositories → kombinle**
3. *Permissions → Repository permissions → Contents*: **Read and write**
4. Oluşan anahtarı kopyala, aşağıdaki kutuya yapıştır (ekranda görünmez, hiçbir yere kaydedilmez).

Dosyalar tek bir kayıtla `app/src/main/assets/` klasörüne gider; GitHub yeni APK'yı ve AAB'yi kendisi derler (5–6 dakika).
Anahtarla uğraşmak istemezsen alttaki hücrede dosyaları indirip GitHub'da elle de yükleyebilirsin.

In [ ]:
import base64, getpass, urllib.request


def github(yontem, yol, anahtar, govde=None):
    istek = urllib.request.Request('https://api.github.com/repos/%s%s' % (REPO, yol), method=yontem,
                                   data=json.dumps(govde).encode() if govde is not None else None,
                                   headers={'Authorization': 'Bearer ' + anahtar, 'Accept': 'application/vnd.github+json',
                                            'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'kombinle-egitim'})
    with urllib.request.urlopen(istek, timeout=300) as y:
        return json.loads(y.read() or b'{}')


def gonder(dosyalar, mesaj):
    anahtar = getpass.getpass('GitHub anahtarı: ').strip()
    if not anahtar:
        print('Anahtar girilmedi; gönderilmedi.')
        return
    son = github('GET', '/git/ref/heads/' + DAL, anahtar)['object']['sha']
    agac = github('GET', '/git/commits/' + son, anahtar)['tree']['sha']
    ogeler = []
    for yerel, uzak in dosyalar:
        blob = github('POST', '/git/blobs', anahtar, {'content': base64.b64encode(open(yerel, 'rb').read()).decode(), 'encoding': 'base64'})
        ogeler.append({'path': uzak, 'mode': '100644', 'type': 'blob', 'sha': blob['sha']})
    yeni_agac = github('POST', '/git/trees', anahtar, {'base_tree': agac, 'tree': ogeler})['sha']
    kayit = github('POST', '/git/commits', anahtar, {'message': mesaj, 'tree': yeni_agac, 'parents': [son]})['sha']
    github('PATCH', '/git/refs/heads/' + DAL, anahtar, {'sha': kayit})
    print('Gönderildi ✓ GitHub yeni APK\'yı derliyor: https://github.com/%s/actions' % REPO)


if SINAMA:
    print('sınama: gönderim atlandı')
else:
    gonder([(MODEL_YOLU, 'app/src/main/assets/kombinle_model.tflite'), (BILGI_YOLU, 'app/src/main/assets/kombinle_model.json')],
           'Kombinle modeli: %d tür, tür %%%.0f, emin olduğunda %%%.0f doğru'
           % (len(SINIF_LISTESI), bilgi['dogruluk']['tur'] * 100, bilgi['dogruluk']['emin75'] * 100))

In [ ]:
# İstersen dosyaları bilgisayarına indir (elle yüklemek için: GitHub → kombinle → app/src/main/assets → Add file → Upload files)
if not SINAMA:
    try:
        from google.colab import files
        files.download(MODEL_YOLU)
        files.download(BILGI_YOLU)
    except Exception as e:
        print('İndirme yalnız Colab\'da çalışır:', e)